In [25]:
import sys
from typing import List, Tuple
import pickle as pkl

import matplotlib.pyplot as plt; plt.rcParams.update({"font.family": "serif"})
import numpy as np
import scipy as sp

import openfermion as of
from openfermionpyscf import run_pyscf

import numpy as np
import openfermion as of

import cirq
import qiskit
import qiskit.qasm2
import quimb.tensor as qtn

sys.path.append("..")
from tensor_network_common import pauli_sum_to_mpo

In [26]:
# Problem to use.
problem: str = "product"  # Options: "hchain", "hubbard", "product", "reactant"

# For Hubbard Hamiltonian.
xdim: int = 4
ydim: int = 3

# For Hydrogen chains.
natoms_list: List[int] = [3, 4, 5, 6, 7]  # Numbers of H atoms in the (linear) chains to study.

hamiltonian_threshold: float = 0.02 # Drop terms in the Hamiltonian with coefficients smaller than this threshold.

# Initial state parameters.
seed: int = 1
alpha: float = 10.0  # The initial state is |ground state> + |random state> / alpha (random state projected to the right particle number).

# Krylov parameters.
subspace_dimension: int = 20  # Fixed Krylov dimension d (must be even; the subspace uses powers -d/2..d/2).
# Set to an integer c to use d = c * natoms instead (c * natoms must be even), or None to use the fixed value above.
subspace_dimension_per_atom: int | None = None  # e.g. 4 for d = 4 N = 2 x number of qubits; None keeps d fixed.

def subspace_dimension_for(natoms: int) -> int:
    """Krylov dimension used for a chain of natoms atoms."""
    if subspace_dimension_per_atom is None:
        return subspace_dimension
    return subspace_dimension_per_atom * natoms

# Results for each choice of d go in their own directory so runs don't overwrite each other.
data_dir: str = "data" if subspace_dimension_per_atom is None else f"data/d{subspace_dimension_per_atom}N"
threshold: float = 1e-4 # 1e-4  # -np.inf  # -1e-4  # Set to -np.inf to disable thresholding.
dt_scale: float = 1.0  # Time step is dt = dt_scale * pi / ||H||_1, with ||H||_1 the sum of |coefficients| (non-identity terms).

# Shot-noise parameters.
nshots_min_exp: float = 2.0  # Shots per estimate are swept over 10^min .. 10^max.
nshots_max_exp: float = 11.0
nshots_per_decade: int = 4
reps: int = 100  # Noise realizations per shot count.
chemical_accuracy: float = 1.6e-3  # Hartree (1 kcal/mol = 1.594 mHa).
noise_threshold_factor: float = 10.0  # For noisy solves, S-threshold = max(threshold, factor * std of S elements). A factor of 1 gave O(1 Ha) errors for H3+.

# Simulation parameters.
max_mpo_bond: int = 128  # Maximum bond dimension for MPO's.
max_mps_bond: int = 256  # Maximum bond dimension for MPS's.

In [194]:
def fill_h_and_s_matrices(
    vectors: List[np.ndarray] | List[np.ndarray],
    mpo: qtn.MatrixProductOperator | np.ndarray,
    verbose: bool = False,
) -> Tuple[np.ndarray, np.ndarray]:
    tensor_algebra = isinstance(mpo, qtn.MatrixProductOperator)

    dim = len(vectors)
    h = np.zeros((dim, dim), dtype=np.complex128)
    s = np.zeros((dim, dim), dtype=np.complex128)

    for i in range(dim):
        for j in range(i, dim):
            if verbose:
                print(f"A_{i}{j}", end=" ")

            if tensor_algebra:
                hij = vectors[i].H @ mpo.apply(vectors[j])
            else:
                hij = vectors[i].conj().T @ mpo @ vectors[j]

            h[i, j] = hij
            if i != j:
                h[j, i] = np.conjugate(hij)

            if tensor_algebra:
                sij = vectors[i].H @ vectors[j]
            else:
                sij = vectors[i].conj().T @ vectors[j]
            s[i, j] = sij
            if i != j:
                s[j, i] = np.conjugate(sij)
    return h, s

# Based on https://quantum.cloud.ibm.com/docs/en/tutorials/krylov-quantum-diagonalization.
# and Algorithm 1.1 of https://arxiv.org/abs/2110.07492.
def solve_regularized_gen_eig(
    h: np.ndarray,
    s: np.ndarray,
    threshold: float = -np.inf,
) -> float:
    s_vals, s_vecs = sp.linalg.eigh(s)
    s_vecs = s_vecs.T
    good_vecs = np.array(
        [vec for val, vec in zip(s_vals, s_vecs) if val > threshold]
    )
    h_reg = good_vecs.conj() @ h @ good_vecs.T
    s_reg = good_vecs.conj() @ s @ good_vecs.T
    return sp.linalg.eigh(h_reg, s_reg)[0][0]

In [195]:
def cirq_pauli_sum_to_qiskit_pauli_op(
    pauli_sum: cirq.PauliSum, qubits: List[cirq.Qid]
) -> qiskit.quantum_info.SparsePauliOp:
    """Returns a qiskit.SparsePauliOp representation of the cirq.PauliSum.

    qubits[i] is mapped to Qiskit qubit i. Qiskit labels are little-endian (the rightmost
    character acts on qubit 0), so the label string is reversed after building it in qubit order.
    """
    cirq_pauli_to_str = {cirq.X: "X", cirq.Y: "Y", cirq.Z: "Z"}

    terms = []
    coeffs = []
    for term in pauli_sum:
        string = ""
        for qubit in qubits:
            if qubit not in term:
                string += "I"
            else:
                string += cirq_pauli_to_str[term[qubit]]
        terms.append(string[::-1])
        assert np.isclose(term.coefficient.imag, 0.0, atol=1e-7)
        coeffs.append(term.coefficient.real)
    return qiskit.quantum_info.SparsePauliOp(terms, coeffs)

In [196]:
def number_projector_mpo(nqubits: int, nparticles: int) -> qtn.MatrixProductOperator:
    """MPO projecting onto computational basis states with Hamming weight nparticles.

    With the Jordan-Wigner transform (OpenFermion convention |1> = occupied), this is the
    projector onto the nparticles-electron sector. The bond index counts particles so far."""
    D = nparticles + 1
    W = np.zeros((D, D, 2, 2))
    for c in range(D):
        for b in range(2):
            if c + b < D:
                W[c, c + b, b, b] = 1.0
    arrays = []
    for i in range(nqubits):
        if i == 0:
            arrays.append(W[0])  # (r, u, d)
        elif i == nqubits - 1:
            arrays.append(W[:, nparticles])  # (l, u, d)
        else:
            arrays.append(W)  # (l, r, u, d)
    return qtn.MatrixProductOperator(arrays)


def one_norm(qubit_operator: of.QubitOperator) -> float:
    """Sum of |coefficients| over non-identity terms (an upper bound on the spectral range / 2)."""
    return sum(abs(c) for term, c in qubit_operator.terms.items() if term != ())

In [197]:
def unitary_h_and_s_variance(
    vectors: List[qtn.MatrixProductState] | List[np.ndarray], ks: List[int], mpo: qtn.MatrixProductOperator | np.ndarray,
    subspace_dimension: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Single-shot variances of the real and imaginary parts of the matrix elements H_kj and S_kj.

    Array index a corresponds to the Krylov power k = a - subspace_dimension // 2.

    With the IBM (Hadamard-test) technique, H_kj ~= <psi0|H U^m|psi0> with m = j - k is estimated
    from <X otimes H> (real part) and <Y otimes H> (imaginary part) using controlled-U^m.
    Since (X otimes H)^2 = (Y otimes H)^2 = I otimes H^2, the single-shot variance is
        <I otimes H^2> - <X otimes H>^2,  with  <I otimes H^2> = 1/2 (<psi0|H^2|psi0> + <psi0|U_m^dag H^2 U_m|psi0>).
    For S_kj set H = I, giving <I otimes I> = 1, so Var = 1 - (Re or Im S)^2.
    The m = 0 elements need no ancilla: H_kk = <psi0|H|psi0> is real with variance
    <H^2> - <H>^2, and S_kk = 1 is known exactly (zero variance).

    The variances only depend on m, consistent with all elements of the same m being estimated
    from the same circuit.

    NOTE: this treats H as a single observable measured in one shot. In practice H is measured
    as several groups of commuting Pauli terms, which gives a larger variance, so shot counts
    from this model are a lower bound.

    Arguments:
    vectors - The vectors U^k|psi0> for all k in ks (must include k = -d..d).
    ks - The power of the unitary generating each vector, e.g. k=2 for U^2|psi_0>.
    Negative values mean backwards time evolution, i.e. powers of U^dagger.
    mpo - Hamiltonian operator.
    subspace_dimension - d; matrices are (d+1) x (d+1).

    Returns:
    h_var - The re/im of h_var[a, b] is the single-shot variance of the re/im of H_ab.
    s_var - The re/im of s_var[a, b] is the single-shot variance of the re/im of S_ab."""
    assert len(ks) == len(vectors)
    assert subspace_dimension % 2 == 0, "subspace_dimension must be even."

    tensor_algebra = isinstance(mpo, qtn.MatrixProductOperator)

    def apply_h(v):
        return mpo.apply(v) if tensor_algebra else mpo @ v

    def inner(u, v):
        return u.H @ v if tensor_algebra else u.conj().T @ v

    psi0 = vectors[ks.index(0)]
    h_psi0 = apply_h(psi0)
    psi0_h2_psi0 = inner(h_psi0, h_psi0).real

    # Toeplitz quantities for each offset m = j - k.
    toeplitz = {}
    for m in range(0, subspace_dimension + 1):
        psi_m = vectors[ks.index(m)]
        h_psi_m = apply_h(psi_m)
        h2_m = inner(h_psi_m, h_psi_m).real
        hm = inner(h_psi0, psi_m)  # <psi0|H U^m|psi0>
        sm = inner(psi0, psi_m)  # <psi0|U^m|psi0>
        exp_h_sq = 0.5 * (psi0_h2_psi0 + h2_m)
        if m == 0:
            hv = complex(psi0_h2_psi0 - hm.real ** 2, 0.0)
            sv = 0.0 + 0.0j
        else:
            hv = (exp_h_sq - hm.real ** 2) + 1.0j * (exp_h_sq - hm.imag ** 2)
            sv = (1.0 - sm.real ** 2) + 1.0j * (1.0 - sm.imag ** 2)
        toeplitz[m] = (hv, sv)

    dim = subspace_dimension + 1
    h_var = np.zeros((dim, dim), dtype=np.complex128)
    s_var = np.zeros((dim, dim), dtype=np.complex128)
    for a in range(dim):
        for b in range(dim):
            hv, sv = toeplitz[abs(b - a)]
            h_var[a, b] = hv
            s_var[a, b] = sv
    return h_var, s_var

In [198]:
def _nonnegative(x: float, what: str) -> float:
    if x < 0.0:
        if abs(x) <= 1e-8:
            return 0.0
        raise ValueError(f"Negative variance for {what}: {x}")
    return x


def perturb_h_s_with_noise(
    h: np.ndarray, s: np.ndarray, h_var: np.ndarray, s_var: np.ndarray, shots: int,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """Add shot noise to H and S, keeping both Hermitian.

    shots is the number of shots per estimate (each Re/Im part of each distinct Toeplitz offset
    m = j - k). Elements with the same m come from the same circuit, so they share one noise draw.
    S_kk = 1 is known exactly and the diagonal of H is real."""
    assert h.shape == h_var.shape, f"{h.shape} != {h_var.shape}"
    assert s.shape == s_var.shape, f"{s.shape} != {s_var.shape}"
    dim = h.shape[0]

    # Standard normal draws for (Re H, Im H, Re S, Im S) for each offset m.
    z = rng.standard_normal((dim, 4))

    noisy_h = np.zeros(h.shape, dtype=complex)
    noisy_s = np.zeros(s.shape, dtype=complex)
    for a in range(dim):
        for b in range(a, dim):
            m = b - a
            std_re_h = np.sqrt(_nonnegative(h_var[a, b].real, f"Re H[{a},{b}]") / shots)
            std_im_h = np.sqrt(_nonnegative(h_var[a, b].imag, f"Im H[{a},{b}]") / shots)
            std_re_s = np.sqrt(_nonnegative(s_var[a, b].real, f"Re S[{a},{b}]") / shots)
            std_im_s = np.sqrt(_nonnegative(s_var[a, b].imag, f"Im S[{a},{b}]") / shots)
            if m == 0:
                noisy_h[a, a] = h[a, a].real + std_re_h * z[m, 0]
                noisy_s[a, a] = s[a, a].real
            else:
                noisy_h[a, b] = h[a, b] + std_re_h * z[m, 0] + 1.0j * std_im_h * z[m, 1]
                noisy_s[a, b] = s[a, b] + std_re_s * z[m, 2] + 1.0j * std_im_s * z[m, 3]
                noisy_h[b, a] = np.conjugate(noisy_h[a, b])
                noisy_s[b, a] = np.conjugate(noisy_s[a, b])

    assert not np.any(np.isnan(noisy_h))
    assert not np.any(np.isnan(noisy_s))
    return noisy_h, noisy_s

In [199]:
def hchain_matrices(natoms: int, subspace_dimension: int, ntrotter: int=1):
    assert subspace_dimension % 2 == 0, "subspace_dimension must be even."
    bond_distance = 1.0
    geometry = [("H", (0, 0, i * bond_distance)) for i in range(natoms)]
    molecule = of.MolecularData(geometry, basis="sto-3g", multiplicity=1, charge=natoms % 2)

    molecule = run_pyscf(molecule, run_scf=True, run_mp2=True, verbose=True)
    nelectrons = molecule.n_electrons
    nqubits = molecule.n_qubits
    qubits = cirq.LineQubit.range(nqubits)

    hamiltonian = of.get_fermion_operator(molecule.get_molecular_hamiltonian())
    hamiltonian = of.jordan_wigner(hamiltonian)
    if hamiltonian_threshold is not None:
        hamiltonian.compress(hamiltonian_threshold)
    nterms = len(hamiltonian.terms)

    hamiltonian_cirq = of.qubit_operator_to_pauli_sum(hamiltonian)
    mpo = pauli_sum_to_mpo(hamiltonian_cirq, qubits, max_bond=max_mpo_bond, verbose=False)

    # Get the ground state in the correct particle-number sector. The qubit Hamiltonian acts on all
    # of Fock space, so add a penalty mu (N - N0)^2 for the DMRG. mu > 2 ||H||_1 guarantees that
    # every other sector is pushed above the target sector.
    hamiltonian_norm = one_norm(hamiltonian)
    penalty_weight = 2.0 * hamiltonian_norm + 1.0
    number_op = of.jordan_wigner(of.number_operator(nqubits))
    penalty = penalty_weight * (number_op - nelectrons * of.QubitOperator(())) ** 2
    penalty.compress()
    penalized_cirq = of.qubit_operator_to_pauli_sum(hamiltonian + penalty)
    penalized_mpo = pauli_sum_to_mpo(penalized_cirq, qubits, max_bond=max_mpo_bond, verbose=False)
    # Start from the Hartree-Fock state (lowest nelectrons spin-orbitals occupied) plus a small random
    # component in the same sector. Pure HF is a fixed point of two-site DMRG (the double excitations
    # that lower its energy act on four sites), and a random start can get stuck in a poor state.
    projector = number_projector_mpo(nqubits, nelectrons)
    hf_state = qtn.MPS_computational_state("1" * nelectrons + "0" * (nqubits - nelectrons))
    dmrg_start = projector.apply(qtn.MPS_rand_state(nqubits, bond_dim=4, seed=seed + 1))
    dmrg_start = hf_state + 0.1 * dmrg_start / dmrg_start.norm()
    dmrg_start /= dmrg_start.norm()
    dmrg = qtn.DMRG2(
        penalized_mpo,
        # Starting the bond-dimension schedule at 2 gets stuck in a local minimum (seen for n=3, 4).
        bond_dims=[2 ** i for i in range(4, int(np.log2(max_mps_bond)) + 1)],
        p0=dmrg_start,
    )
    converged = dmrg.solve(verbosity=1, max_sweeps=50)
    if not converged:
        print("WARNING: DMRG did not converge.")

    ground_state = dmrg.state
    ground_state /= ground_state.norm()
    energy_exact = np.real(ground_state.H @ mpo.apply(ground_state))
    print(f"Ground state energy in N={nelectrons} sector: {energy_exact}")
    if nqubits <= 14:  # Cross-check against exact diagonalization in the particle-number sector.
        h_sparse = of.get_sparse_operator(hamiltonian, n_qubits=nqubits)
        occupations = np.array([bin(i).count("1") for i in range(2 ** nqubits)])
        sector = np.where(occupations == nelectrons)[0]
        energy_ed = np.linalg.eigvalsh(h_sparse[sector][:, sector].toarray())[0]
        print(f"Exact diagonalization energy: {energy_ed}")
        assert np.isclose(energy_exact, energy_ed, atol=1e-6), "DMRG did not find the sector ground state."

    sector_weight = np.real(ground_state.H @ projector.apply(ground_state))
    print(f"Ground state weight in N={nelectrons} sector: {sector_weight}")

    # Random perturbation, projected onto the same particle-number sector.
    random_state = projector.apply(qtn.MPS_rand_state(mpo.L, bond_dim=2, seed=seed))
    random_state /= random_state.norm()
    bvec = ground_state + random_state / alpha
    bvec /= bvec.norm()

    # Time step from the 1-norm of H (upper bound on the spectral norm), which scales
    # polynomially with system size (the Frobenius norm mpo.norm() scales like 2^(n/2)).
    dt = dt_scale * np.pi / hamiltonian_norm
    print(f"||H||_1 = {hamiltonian_norm}, dt = {dt}")

    hamiltonian_qiskit = cirq_pauli_sum_to_qiskit_pauli_op(hamiltonian_cirq, qubits)
    trotter_operation = qiskit.circuit.library.PauliEvolutionGate(
        hamiltonian_qiskit,
        time=dt,
        synthesis=qiskit.synthesis.LieTrotter(reps=ntrotter)
    )
    trotter_circuit = qiskit.QuantumCircuit(nqubits)
    trotter_circuit.append(trotter_operation, trotter_circuit.qubits)
    trotter_circuit = qiskit.transpile(
        trotter_circuit, basis_gates=["u3", "cx"]
    )  # TODO: Compile to a target backend, e.g. IBM Fez.
    # print("Trotter circuit gate counts:")
    # print(trotter_circuit.count_ops())

    forward = trotter_circuit.copy()
    forward_qasm = qiskit.qasm2.dumps(forward)

    reverse = trotter_circuit.inverse().copy()
    reverse_qasm = qiskit.qasm2.dumps(reverse)

    forward_vectors = [bvec]

    for k in range(subspace_dimension):
        # print("Forward:", k)
        new = qtn.CircuitMPS.from_openqasm2_str(
            forward_qasm, psi0=forward_vectors[-1], progbar=False, max_bond=max_mps_bond,
        ).psi
        forward_vectors.append(new)

    reverse_vectors = [bvec]
    for k in range(subspace_dimension):
        # print("Reverse:", k)
        new = qtn.CircuitMPS.from_openqasm2_str(
            reverse_qasm, psi0=reverse_vectors[-1], progbar=False, max_bond=max_mps_bond,
        ).psi
        reverse_vectors.append(new)

    _ = reverse_vectors.pop(0)
    reverse_vectors = list(reversed(reverse_vectors))

    all_vectors_u_trotter = reverse_vectors + forward_vectors
    ks = list(range(-subspace_dimension, subspace_dimension + 1))

    # Trotterized evolution need not conserve particle number exactly; report the leakage.
    last = forward_vectors[-1]
    leakage = 1.0 - np.real(last.H @ projector.apply(last)) / np.real(last.H @ last)
    print(f"Particle-number leakage after {subspace_dimension} Trotter steps: {leakage:.3e}")

    vectors_u_trotter = []
    for k, vec in zip(ks, all_vectors_u_trotter):
        if (k >= -subspace_dimension // 2) and (k <= subspace_dimension // 2):
            vectors_u_trotter.append(vec)

    h, s = fill_h_and_s_matrices(vectors_u_trotter, mpo, verbose=False)
    h_var, s_var = unitary_h_and_s_variance(all_vectors_u_trotter, ks, mpo, subspace_dimension)
    return h, s, h_var, s_var, energy_exact

In [200]:
def run_natoms(natoms: int, subspace_dimension: int, reps: int=10):
    h, s, h_var, s_var, energy_exact = hchain_matrices(natoms, subspace_dimension, ntrotter=3)
    subspace_energy = solve_regularized_gen_eig(h, s, threshold=threshold)
    print(f"Noiseless Krylov energy: {subspace_energy}, error: {abs(subspace_energy - energy_exact)}")
    rng = np.random.default_rng(seed)
    num = int(round((nshots_max_exp - nshots_min_exp) * nshots_per_decade)) + 1
    nshots = np.logspace(nshots_min_exp, nshots_max_exp, num=num)
    # Distinct estimates: m = 0 (Re H only), and m = 1..d (Re/Im of H and S).
    estimates_per_shot_count = 1 + 4 * subspace_dimension
    max_s_var = max(s_var.real.max(), s_var.imag.max())
    noisy_energies = np.zeros((len(nshots), reps))
    noisy_thresholds = np.zeros(len(nshots))
    for i, shots in enumerate(nshots):
        noisy_threshold = max(threshold, noise_threshold_factor * np.sqrt(max_s_var / shots))
        noisy_thresholds[i] = noisy_threshold
        for r in range(reps):
            noisy_h, noisy_s = perturb_h_s_with_noise(h, s, h_var, s_var, shots, rng)
            noisy_energies[i, r] = solve_regularized_gen_eig(noisy_h, noisy_s, noisy_threshold)
    errs = np.abs(noisy_energies - energy_exact)
    result = {
        "natoms": natoms,
        "energy_exact": energy_exact,
        "h": h,
        "s": s,
        "h_var": h_var,
        "s_var": s_var,
        "nshots": nshots,  # Shots per Re/Im estimate of each distinct Toeplitz element.
        "total_shots": nshots * estimates_per_shot_count,
        "estimates_per_shot_count": estimates_per_shot_count,
        "noisy_energies": noisy_energies,  # Shape (len(nshots), reps).
        "noisy_thresholds": noisy_thresholds,
        "mean_errs": errs.mean(axis=1),
        "std_errs": errs.std(axis=1),
        "subspace_energy": subspace_energy,
        "subspace_error": abs(subspace_energy - energy_exact),
        "params": {
            "subspace_dimension": subspace_dimension,
            "ntrotter": 3,
            "hamiltonian_threshold": hamiltonian_threshold,
            "seed": seed,
            "alpha": alpha,
            "threshold": threshold,
            "dt_scale": dt_scale,
            "noise_threshold_factor": noise_threshold_factor,
            "reps": reps,
            "max_mpo_bond": max_mpo_bond,
            "max_mps_bond": max_mps_bond,
            "bond_distance": 1.0,
            "basis": "sto-3g",
        },
    }
    return result

In [5]:
import os
os.makedirs(data_dir, exist_ok=True)

results = {}
for n in natoms_list:
    print(f"===== natoms = {n}, d = {subspace_dimension_for(n)} =====")
    results[n] = run_natoms(n, subspace_dimension_for(n), reps=reps)
    with open(f"{data_dir}/results_n{n}.pkl", "wb") as f:
        pkl.dump(results[n], f)

NameError: name 'natoms_list' is not defined

In [27]:
# Reload saved results (this cell and below can be run without re-running the simulations).
results = {}
for n in natoms_list:
    with open(f"{data_dir}/results_n{n}.pkl", "rb") as f:
        results[n] = pkl.load(f)

In [28]:
# NOTE: the mean error is dominated by rare spurious eigenvalues far below the exact energy
# (a few % of realizations at intermediate shot counts), so use the median / 95th percentile.
def shots_for_accuracy(nshots: np.ndarray, err_stat: np.ndarray, target: float) -> float:
    """Smallest shot count (log-log interpolated) above which err_stat stays <= target.

    Returns np.inf if the target is not reached at the largest shot count, and np.nan if it is
    already reached at the smallest one (i.e. only an upper bound nshots[0] is known)."""
    ok = err_stat <= target
    if not ok[-1]:
        return np.inf
    # Last index that fails; everything after it succeeds.
    fails = np.where(~ok)[0]
    if len(fails) == 0:
        return np.nan
    i = fails[-1]
    x0, x1 = np.log10(nshots[i]), np.log10(nshots[i + 1])
    y0, y1 = np.log10(err_stat[i]), np.log10(err_stat[i + 1])
    t = (np.log10(target) - y0) / (y1 - y0)
    return 10 ** (x0 + t * (x1 - x0))

In [29]:
summary = []
for n, res in results.items():
    errs = np.abs(res["noisy_energies"] - res["energy_exact"])
    row = {
        "natoms": n,
        "energy_exact": res["energy_exact"],
        "subspace_error": res["subspace_error"],
    }
    for name, stat in [("mean", errs.mean(axis=1)),
                       ("median", np.median(errs, axis=1)),
                       ("p95", np.percentile(errs, 95, axis=1))]:
        shots = shots_for_accuracy(res["nshots"], stat, chemical_accuracy)
        row[f"shots_{name}"] = shots
        row[f"total_shots_{name}"] = shots * res["estimates_per_shot_count"]
    summary.append(row)

print(f"Chemical accuracy target: {chemical_accuracy} Ha. Shots are per Re/Im estimate; total = x (1 + 4d). Data: {data_dir}")
print(f"{'N':>3} {'E_exact':>12} {'Krylov err':>11} {'mean':>10} {'median':>10} {'p95':>10} {'total(p95)':>11}")
for row in summary:
    print(f"{row['natoms']:>3} {row['energy_exact']:>12.6f} {row['subspace_error']:>11.2e} "
          f"{row['shots_mean']:>10.2e} {row['shots_median']:>10.2e} {row['shots_p95']:>10.2e} {row['total_shots_p95']:>11.2e}")

with open(f"{data_dir}/chemical_accuracy_summary.pkl", "wb") as f:
    pkl.dump({"chemical_accuracy": chemical_accuracy, "summary": summary}, f)

Chemical accuracy target: 0.0016 Ha. Shots are per Re/Im estimate; total = x (1 + 4d). Data: data
  N      E_exact  Krylov err       mean     median        p95  total(p95)
  3    -1.224063    3.43e-04   2.33e+06   2.98e+03   1.57e+06    1.27e+08
  4    -2.168871    5.62e-04   3.89e+09   4.14e+04   5.08e+05    4.11e+07
  5    -2.361819    2.41e-04   1.47e+05   4.69e+04   2.91e+05    2.35e+07
  6    -3.191824    9.94e-05   3.95e+05   4.80e+04   5.92e+05    4.79e+07
  7    -3.422412    1.20e-03   2.55e+05   1.81e+05   5.10e+06    4.13e+08


In [30]:
fig, ax = plt.subplots()
for n, res in results.items():
    errs = np.abs(res["noisy_energies"] - res["energy_exact"])
    line, = ax.plot(res["nshots"], np.median(errs, axis=1), "o-", ms=3, label=f"N={n}")
    ax.fill_between(res["nshots"], np.percentile(errs, 5, axis=1), np.percentile(errs, 95, axis=1),
                    color=line.get_color(), alpha=0.2)
    ax.axhline(res["subspace_error"], color=line.get_color(), linestyle=":", lw=1)
ax.axhline(chemical_accuracy, color="black", linestyle="--", label="Chemical accuracy")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Shots per matrix-element estimate")
ax.set_ylabel("Absolute energy error (Ha)")
ax.set_title("Lines: median; bands: 5-95th percentile; dotted: noiseless Krylov", fontsize=10)
ax.legend()
fig.savefig(f"{data_dir}/error_vs_shots.pdf", bbox_inches="tight")

In [31]:
fig, ax = plt.subplots()
ns = [row["natoms"] for row in summary]
ax.plot(ns, [row["shots_median"] for row in summary], "o-", label="Median error")
ax.plot(ns, [row["shots_p95"] for row in summary], "^--", label="95th-percentile error")
ax.set_yscale("log")
ax.set_xticks(ns)
ax.set_xlabel("Number of H atoms")
ax.set_ylabel("Shots per estimate for chemical accuracy")
ax.legend(title="Error at or below 1.6 mHa")
fig.savefig(f"{data_dir}/shots_vs_natoms.pdf", bbox_inches="tight")

## Extrapolating the shot count to larger chains

First-order (linear-response) model of the shot noise. For a perturbation $(\delta H, \delta S)$ of the
thresholded generalized eigenproblem,
$$\delta E \approx x^\dagger(\delta H - E\,\delta S)\,x, \qquad x^\dagger S x = 1,$$
with $x$ the noiseless ground eigenvector. With the noise model of `perturb_h_s_with_noise` (one draw per
Toeplitz offset, independent Re/Im parts, independent $H$ and $S$), $\delta E = \sum_i c_i z_i / \sqrt{M}$ for
independent standard normals $z_i$ and $M$ shots per estimate, so the single-shot variance is
$\sigma_1^2 = \sum_i c_i^2$. The S-threshold depends on $M$, so $x$, $E$ and the thresholding bias are recomputed
for every $M$, and the predicted error is $|\mathrm{bias} + \sigma_1 z/\sqrt{M}|$.

$\sigma_1^2$ is evaluated at the threshold that applies where the median error reaches chemical accuracy,
then extrapolated in $N$ three ways: $\kappa E^2$ (with $E$ from CCSD(T) beyond the simulated sizes),
a power law, and an exponential. Shots for zero bias are $M = q^2 \sigma_1^2 / \varepsilon^2$ with $q$ the
median (0.674) or 95th percentile (1.96) of $|z|$.

Requires the imports, the parameters, and the reload/analysis cells above (`results`, `summary`,
`shots_for_accuracy`).

In [32]:
# Extrapolation parameters.
fit_natoms_min: int = 4  # Smallest N used in the fits (N=3 has a much larger S-threshold near chemical accuracy).
extrapolate_natoms: int = 22  # Chain length to extrapolate to.
first_order_shots = np.logspace(nshots_min_exp, nshots_max_exp, 181)  # Dense shot grid for the first-order model.
z_samples = np.random.default_rng(0).standard_normal(200_000)  # For quantiles of |bias + sigma z|.

# Colors (one per extrapolation model, consistent across plots).
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#1f1f1e", "#8a897f"
extrap_rc = {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
             "grid.color": "#e4e3dd", "grid.linewidth": 0.6, "axes.axisbelow": True}

In [33]:
def thresholded_ground(h: np.ndarray, s: np.ndarray, threshold: float) -> Tuple[float, np.ndarray]:
    """Ground eigenpair of the thresholded problem; x is in the full basis with x^dag S x = 1."""
    s_vals, s_vecs = sp.linalg.eigh(s)
    V = s_vecs[:, s_vals > threshold]
    e, y = sp.linalg.eigh(V.conj().T @ h @ V, V.conj().T @ s @ V)
    x = V @ y[:, 0]
    x /= np.sqrt(np.real(x.conj() @ s @ x))
    return e[0], x


def first_order_sigma1_sq(h, s, h_var, s_var, threshold: float):
    """Single-shot variance of the energy to first order in the noise, and its split by noise source."""
    E, x = thresholded_ground(h, s, threshold)
    dim = h.shape[0]
    w = np.outer(x.conj(), x)  # w[a, b] = x_a^* x_b
    contributions = {"H_re": 0.0, "H_im": 0.0, "S_re": 0.0, "S_im": 0.0, "H_diag": 0.0}
    # Diagonal (m = 0): one shared real draw for H; S_kk is exact.
    contributions["H_diag"] = max(h_var[0, 0].real, 0.0) * np.real(np.trace(w)) ** 2
    for m in range(1, dim):
        a = np.arange(dim - m)
        b = a + m
        # Pair (a,b) & (b,a) contributes 2 Re(w_ab dX_ab), with dX_ab = sigma_re z_re + i sigma_im z_im.
        sum_re = 2.0 * np.sum(np.real(w[a, b]))
        sum_im = -2.0 * np.sum(np.imag(w[a, b]))
        contributions["H_re"] += max(h_var[0, m].real, 0.0) * sum_re ** 2
        contributions["H_im"] += max(h_var[0, m].imag, 0.0) * sum_im ** 2
        contributions["S_re"] += E ** 2 * max(s_var[0, m].real, 0.0) * sum_re ** 2
        contributions["S_im"] += E ** 2 * max(s_var[0, m].imag, 0.0) * sum_im ** 2
    return E, sum(contributions.values()), contributions


def noisy_threshold_for(res: dict, shots: float) -> float:
    """S-threshold used by run_natoms at this shot count."""
    p = res["params"]
    max_s_var = max(res["s_var"].real.max(), res["s_var"].imag.max())
    return max(p["threshold"], p["noise_threshold_factor"] * np.sqrt(max_s_var / shots))

In [34]:
first_order = []
for n, res in results.items():
    h, s, h_var, s_var, E0 = res["h"], res["s"], res["h_var"], res["s_var"], res["energy_exact"]
    med, p95 = [], []
    for M in first_order_shots:
        E, s1sq, _ = first_order_sigma1_sq(h, s, h_var, s_var, noisy_threshold_for(res, M))
        errs = np.abs((E - E0) + np.sqrt(s1sq / M) * z_samples)
        med.append(np.median(errs))
        p95.append(np.percentile(errs, 95))
    med, p95 = np.array(med), np.array(p95)
    shots_med = shots_for_accuracy(first_order_shots, med, chemical_accuracy)
    thr_star = noisy_threshold_for(res, shots_med)
    E_star, s1sq_star, contrib = first_order_sigma1_sq(h, s, h_var, s_var, thr_star)
    first_order.append({
        "natoms": n,
        "E": E0,
        "model_shots": first_order_shots,
        "model_median_err": med,
        "model_p95_err": p95,
        "shots_median": shots_med,
        "shots_p95": shots_for_accuracy(first_order_shots, p95, chemical_accuracy),
        "threshold_star": thr_star,
        "sigma1sq_star": s1sq_star,
        "bias_star": E_star - E0,
        "contributions": contrib,
    })

print(f"{'N':>2} {'sigma1^2':>9} {'sigma1^2/E^2':>12} {'thr*':>8} {'bias* (mHa)':>11} {'1st med':>9} {'MC med':>9}")
mc = {row["natoms"]: row for row in summary}
for r in first_order:
    print(f"{r['natoms']:>2} {r['sigma1sq_star']:>9.3f} {r['sigma1sq_star'] / r['E'] ** 2:>12.4f} {r['threshold_star']:>8.1e} "
          f"{r['bias_star'] * 1e3:>11.2f} {r['shots_median']:>9.2e} {mc[r['natoms']]['shots_median']:>9.2e}")

with open(f"{data_dir}/first_order_sigma1sq.pkl", "wb") as f:
    pkl.dump(first_order, f)

 N  sigma1^2 sigma1^2/E^2     thr* bias* (mHa)   1st med    MC med
 3     0.011       0.0072  1.8e-01        1.32  3.12e+03  2.98e+03
 4     0.169       0.0359  5.3e-02        0.86  3.51e+04  4.14e+04
 5     0.195       0.0349  5.1e-02        0.69  3.83e+04  4.69e+04
 6     0.346       0.0339  4.0e-02        0.23  6.25e+04  4.80e+04
 7     0.414       0.0354  3.3e-02       -0.98  9.02e+04  1.81e+05


In [35]:
# CCSD(T) energies (full Hamiltonian, same geometry and charge as hchain_matrices) for E(N) beyond the
# simulated sizes. Cached because they are reused by the plots.
from pyscf import gto, scf, cc

ccsdt_path = "data/ccsdt_energies.pkl"
ccsdt_natoms = list(range(3, extrapolate_natoms + 1))
try:
    with open(ccsdt_path, "rb") as f:
        ccsdt_energies = pkl.load(f)
except FileNotFoundError:
    ccsdt_energies = {}
for n in ccsdt_natoms:
    if n not in ccsdt_energies:
        mol = gto.M(atom=[("H", (0, 0, i * 1.0)) for i in range(n)], basis="sto-3g", charge=n % 2, spin=0, verbose=0)
        mf = scf.RHF(mol).run()
        mycc = cc.CCSD(mf).run()
        ccsdt_energies[n] = mycc.e_tot + mycc.ccsd_t()
with open(ccsdt_path, "wb") as f:
    pkl.dump(ccsdt_energies, f)
print({n: round(e, 4) for n, e in sorted(ccsdt_energies.items())})

{3: np.float64(-1.2249), 4: np.float64(-2.1664), 5: np.float64(-2.3678), 6: np.float64(-3.2361), 7: np.float64(-3.4824), 8: np.float64(-4.3075), 9: np.float64(-4.5834), 10: np.float64(-5.3798), 11: np.float64(-5.6767), 12: np.float64(-6.4525), 13: np.float64(-6.7652), 14: np.float64(-7.5255), 15: np.float64(-7.8504), 16: np.float64(-8.5986), 17: np.float64(-8.9333), 18: np.float64(-9.6718), 19: np.float64(-10.0145), 20: np.float64(-10.7451), 21: np.float64(-11.0945), 22: np.float64(-11.8184)}


In [36]:
fit_rows = [r for r in first_order if r["natoms"] >= fit_natoms_min]
N_fit = np.array([r["natoms"] for r in fit_rows])
s1_fit = np.array([r["sigma1sq_star"] for r in fit_rows])
E_fit = np.array([r["E"] for r in fit_rows])

kappa = np.mean(s1_fit / E_fit ** 2)
kappa_std = np.std(s1_fit / E_fit ** 2)
powerlaw = np.polyfit(np.log(N_fit), np.log(s1_fit), 1)
exponential = np.polyfit(N_fit, np.log(s1_fit), 1)

# E(N): simulated (exact, truncated H) where available, CCSD(T) (full H) beyond.
E_of_N = {n: e for n, e in ccsdt_energies.items()}
E_of_N.update({r["natoms"]: r["E"] for r in first_order})

N_curve_int = np.arange(fit_natoms_min, extrapolate_natoms + 1)
N_curve = np.linspace(fit_natoms_min, extrapolate_natoms, 200)
extrapolation_models = [
    (f"$\\kappa E^2$, $\\kappa$={kappa:.3f} (preferred)", BLUE, N_curve_int,
     kappa * np.array([E_of_N[n] ** 2 for n in N_curve_int])),
    (f"Power law $\\propto N^{{{powerlaw[0]:.2f}}}$", ORANGE, N_curve, np.exp(np.polyval(powerlaw, np.log(N_curve)))),
    (f"Exponential $\\propto e^{{{exponential[0]:.2f}N}}$", AQUA, N_curve, np.exp(np.polyval(exponential, N_curve))),
]

Q50, Q95 = 0.6745, 1.960  # Median and 95th percentile of |N(0, 1)| (zero bias).
estimates = 1 + 4 * subspace_dimension_for(extrapolate_natoms)  # Distinct estimates at the target size.
print(f"kappa = {kappa:.4f} +/- {kappa_std:.4f} (N = {fit_natoms_min}..{max(N_fit)})")
print(f"Shots at N = {extrapolate_natoms} (zero bias, d = {subspace_dimension_for(extrapolate_natoms)}). Total = per-estimate x {estimates}.")
print(f"{'model':40s} {'sigma1^2':>9} {'median/est':>11} {'95%/est':>10} {'median tot':>11} {'95% tot':>10}")
extrapolation_table = []
for label, _, _, y in extrapolation_models:
    s1 = y[-1]
    m50, m95 = Q50 ** 2 * s1 / chemical_accuracy ** 2, Q95 ** 2 * s1 / chemical_accuracy ** 2
    extrapolation_table.append({"model": label, "sigma1sq": s1, "shots_median": m50, "shots_p95": m95})
    print(f"{label:40s} {s1:>9.2f} {m50:>11.2e} {m95:>10.2e} {m50 * estimates:>11.2e} {m95 * estimates:>10.2e}")

with open(f"{data_dir}/shot_extrapolation.pkl", "wb") as f:
    pkl.dump({"kappa": kappa, "kappa_std": kappa_std, "powerlaw": powerlaw, "exponential": exponential,
              "fit_natoms_min": fit_natoms_min, "extrapolate_natoms": extrapolate_natoms,
              "E_of_N": E_of_N, "table": extrapolation_table}, f)

kappa = 0.0350 +/- 0.0007 (N = 4..7)
Shots at N = 22 (zero bias, d = 20). Total = per-estimate x 81.
model                                     sigma1^2  median/est    95%/est  median tot    95% tot
$\kappa E^2$, $\kappa$=0.035 (preferred)      4.89    8.69e+05   7.34e+06    7.04e+07   5.94e+08
Power law $\propto N^{1.74}$                  3.03    5.38e+05   4.54e+06    4.36e+07   3.68e+08
Exponential $\propto e^{0.33N}$              57.78    1.03e+07   8.67e+07    8.32e+08   7.02e+09


In [37]:
with plt.rc_context(extrap_rc):
    N_all = np.array([r["natoms"] for r in first_order])
    ratio = np.array([r["sigma1sq_star"] / r["E"] ** 2 for r in first_order])
    fitted = N_all >= fit_natoms_min

    fig, ax = plt.subplots(figsize=(5.6, 3.8))
    ax.axhspan(kappa - kappa_std, kappa + kappa_std, color=BLUE, alpha=0.12, lw=0)
    ax.axhline(kappa, color=BLUE, lw=2,
               label=f"Mean over N = {fit_natoms_min}-{max(N_fit)}: $\\kappa$ = {kappa:.4f} $\\pm$ {kappa_std:.4f}")
    ax.plot(N_all[fitted], ratio[fitted], "o", color=INK, ms=8, mec="white", mew=1.5, zorder=5, label="Notebook data (fitted)")
    ax.plot(N_all[~fitted], ratio[~fitted], "o", mfc="white", mec=INK, ms=8, zorder=5, label="Excluded from fit")
    for n, q in zip(N_all, ratio):
        ax.annotate(f"{q:.4f}", (n, q), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8, color=INK)
    ax.set_xticks(N_all)
    ax.set_ylim(0, 1.3 * ratio.max())
    ax.set_xlim(N_all.min() - 0.5, N_all.max() + 0.5)
    ax.set_xlabel("Number of H atoms, N")
    ax.set_ylabel("$\\sigma_1^2 / E^2$")
    ax.set_title("Single-shot variance relative to $E^2$", loc="left", fontsize=12)
    ax.legend(fontsize=8, frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(f"{data_dir}/sigma1sq_over_E2.pdf")
    fig.show()

/var/folders/62/vpq4vzfx3zlfyyr3mw8zdmrc0000gn/T/ipykernel_30475/2939687676.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


In [38]:
with plt.rc_context(extrap_rc):
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for label, color, x, y in extrapolation_models:
        ax.plot(x, y, color=color, lw=2, label=label)
        ax.plot(x[-1], y[-1], "o", color=color, ms=8, mec="white", mew=1.5, zorder=4)
        ax.annotate(f"{y[-1]:.1f}", (x[-1], y[-1]), xytext=(7, 0), textcoords="offset points", va="center", fontsize=9, color=INK)
    excluded = [r for r in first_order if r["natoms"] < fit_natoms_min]
    ax.plot(N_fit, s1_fit, "o", color=INK, ms=8, mec="white", mew=1.5, zorder=5,
            label="First-order $\\sigma_1^2$, notebook data (fitted)")
    if excluded:
        ax.plot([r["natoms"] for r in excluded], [r["sigma1sq_star"] for r in excluded], "o", mfc="white", mec=INK, ms=8,
                zorder=5, label="Excluded from fit (large S threshold)")
    ax.axvline(extrapolate_natoms, color=MUTED, ls=":", lw=1)
    ax.set_yscale("log")
    ax.set_xlim(min(natoms_list) - 0.5, extrapolate_natoms + 1.8)
    ax.set_xticks(sorted(set(natoms_list) | set(range(10, extrapolate_natoms + 1, 4)) | {extrapolate_natoms}))
    ax.set_xlabel("Number of H atoms, N")
    ax.set_ylabel("Single-shot energy variance $\\sigma_1^2$ (Ha$^2$)")
    ax.set_title(f"Extrapolating $\\sigma_1^2$ from N = {fit_natoms_min}-{max(N_fit)}", loc="left", fontsize=12)
    ax.legend(fontsize=8, frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(f"{data_dir}/sigma1sq_extrapolation.pdf")

In [39]:
from matplotlib.lines import Line2D

with plt.rc_context(extrap_rc):
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    end_values = []
    for label, color, x, y in extrapolation_models:
        for q, ls in [(Q50, "-"), (Q95, "--")]:
            shots = q ** 2 * y / chemical_accuracy ** 2
            ax.plot(x, shots, color=color, lw=2, ls=ls)
            ax.plot(x[-1], shots[-1], "o", color=color, ms=7, mec="white", mew=1.5, zorder=4)
            end_values.append(shots[-1])
    # End labels, spaced at least 0.3 decades apart so they never collide.
    placed = []
    for y in sorted(end_values):
        ly = np.log10(y)
        if placed and ly - placed[-1] < 0.3:
            ly = placed[-1] + 0.3
        placed.append(ly)
        ax.annotate(f"{y:.1e}", (extrapolate_natoms, y), xytext=(extrapolate_natoms + 0.35, 10 ** ly),
                    textcoords="data", va="center", fontsize=8, color=INK)

    N_mc = [row["natoms"] for row in summary]
    ax.plot(N_mc, [row["shots_median"] for row in summary], "o", color=INK, ms=8, mec="white", mew=1.5, zorder=5)
    ax.plot(N_mc, [row["shots_p95"] for row in summary], "^", color=INK, ms=8, mec="white", mew=1.5, zorder=5)
    ax.axvline(extrapolate_natoms, color=MUTED, ls=":", lw=1)

    handles = [Line2D([], [], color=c, lw=2, label=l) for l, c, _, _ in extrapolation_models]
    handles += [Line2D([], [], color=MUTED, lw=2, ls="-", label="Extrapolated median"),
                Line2D([], [], color=MUTED, lw=2, ls="--", label="Extrapolated 95% of runs"),
                Line2D([], [], color=INK, marker="o", ls="", ms=7, label="Notebook Monte Carlo, median"),
                Line2D([], [], color=INK, marker="^", ls="", ms=7, label="Notebook Monte Carlo, 95% of runs")]
    ax.legend(handles=handles, fontsize=7.5, frameon=False, loc="upper left", ncol=2)
    ax.set_yscale("log")
    ax.set_ylim(1e3, 1e10)
    ax.set_xlim(min(natoms_list) - 0.5, extrapolate_natoms + 2.5)
    ax.set_xticks(sorted(set(natoms_list) | set(range(10, extrapolate_natoms + 1, 4)) | {extrapolate_natoms}))
    ax.set_xlabel("Number of H atoms, N")
    ax.set_ylabel("Shots per matrix-element estimate")
    ax.set_title(f"Shots for chemical accuracy ({chemical_accuracy * 1e3:.1f} mHa)", loc="left", fontsize=12)
    d_rule = (f"d = {subspace_dimension}" if subspace_dimension_per_atom is None
              else f"d = {subspace_dimension_per_atom}N, i.e. {subspace_dimension_for(extrapolate_natoms)} at N = {extrapolate_natoms}")
    ax.text(0.0, -0.17, f"Total shots = (1 + 4d) x per-estimate value ({d_rule}: x{estimates}). "
            "Extrapolation assumes zero bias.", transform=ax.transAxes, fontsize=8, color=MUTED)
    fig.tight_layout()
    fig.savefig(f"{data_dir}/shots_extrapolation.pdf")

## Norm-based shot bound, and comparison of fixed-$d$ and $d \propto N$ runs

**Bound.** To first order in the noise, $|\delta E| \le \|x\|^2 \left(\|\delta H\|_2 + |E|\,\|\delta S\|_2\right)$ with $x^\dagger S x = 1$.
Two patterns in the data make every factor computable at large $N$:

* Per-element variances: $\mathrm{Var}(\mathrm{Re/Im}\,H_m) = E^2\,\mathrm{Var}(\mathrm{Re/Im}\,S_m) + O(0.1)$, so
  $\mathrm{Var}(H_m) \le \langle H^2\rangle = E^2 + \mathrm{Var}_{\psi_0}(H)$ and $\mathrm{Var}(S_m) \le 1$
  (observed: $\mathrm{Var}(\mathrm{Re}\,S_m) + \mathrm{Var}(\mathrm{Im}\,S_m) \approx 1$).
* $\|x\|^2 \approx 1/(d+1)$, independent of $N$ (the Krylov vectors are all close to $\psi_0$ up to a phase).

The noise matrices are Hermitian Toeplitz with one draw per offset, so $\|\delta X\|_2 = \sigma\,\|G_{d+1}\|_2/\sqrt{M}$, giving
$$M \ge \left[\frac{\|x\|^2\left(\sqrt{\langle H^2\rangle} + |E|\right)\|G_{d+1}\|_2}{\varepsilon - |\mathrm{bias}|}\right]^2.$$
The median / 95th percentile of $\|G_{d+1}\|_2$ give the median / 95% bound. At the simulated sizes the bound uses the actual
$\|x\|^2$, $\langle H^2\rangle$ and thresholding bias; the extrapolation uses $\|x\|^2 = 1/(d+1)$ and zero bias.

**Extrapolations of $\sigma_1^2$** (first order, evaluated at the threshold that applies at chemical accuracy):
$\kappa E^2/(d+1)$ with $\kappa = \langle \sigma_1^2 (d+1)/E^2\rangle$ (constant across both runs), a power law and an exponential in $N$.
Shots per estimate are $q^2\sigma_1^2/\varepsilon^2$ with $q$ = 0.674 (median) or 1.96 (95%); total shots are $(1 + 4d)\times$ that.

Everything is computed for each dataset in `bound_datasets` and saved, so the plot cells only read the saved files.
Requires the imports and parameters cells, `shots_for_accuracy`, the first-order functions (`thresholded_ground`,
`first_order_sigma1_sq`, `noisy_threshold_for`) and the CCSD(T) energies in `data/ccsdt_energies.pkl`.

In [40]:
# Datasets to analyse: label -> directory and the rule that was used for d.
bound_datasets = {
    "d = 20": {"dir": "data", "d_fixed": 20, "d_per_atom": None},
    "d = 4N": {"dir": "data/d4N", "d_fixed": None, "d_per_atom": 4},
}
bound_fit_natoms_min: int = 4  # Smallest N used in the fits.
bound_extrapolate_natoms: int = 22
var_psi0_extrap: float = 0.1  # Var_psi0(H) used beyond the simulated sizes (observed <= 0.08 Ha^2).
toeplitz_ndraw: int = 4000  # Draws for the ||G_D||_2 quantiles (cached in data/toeplitz_norm_quantiles.pkl).
Q50, Q95 = 0.6745, 1.960  # Median and 95th percentile of |N(0, 1)|.

# Colors: one per extrapolation model, consistent across plots.
BLUE, ORANGE, AQUA, VIOLET, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#1f1f1e", "#8a897f"
bound_rc = {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
            "grid.color": "#e4e3dd", "grid.linewidth": 0.6, "axes.axisbelow": True}


def d_for(dataset: dict, natoms: int) -> int:
    """Krylov dimension used by a dataset at a given chain length."""
    return dataset["d_fixed"] if dataset["d_per_atom"] is None else dataset["d_per_atom"] * natoms

In [41]:
import glob
import re

toeplitz_cache_path = "data/toeplitz_norm_quantiles.pkl"
try:
    with open(toeplitz_cache_path, "rb") as f:
        toeplitz_quantiles = pkl.load(f)
except FileNotFoundError:
    toeplitz_quantiles = {}


def toeplitz_norm_quantiles(D: int) -> Tuple[float, float]:
    """Median and 95th percentile of ||G_D||_2 for Hermitian Toeplitz G with one complex standard normal
    per offset (Re, Im variance 1/2 each, real diagonal), as in perturb_h_s_with_noise. Cached on disk."""
    if D not in toeplitz_quantiles:
        rng = np.random.default_rng(D)
        norms = np.empty(toeplitz_ndraw)
        for i in range(toeplitz_ndraw):
            c = np.sqrt(0.5) * (rng.standard_normal(D) + 1j * rng.standard_normal(D))
            c[0] = np.sqrt(0.5) * rng.standard_normal()
            norms[i] = np.linalg.norm(sp.linalg.toeplitz(np.conj(c), c), 2)
        toeplitz_quantiles[D] = (np.median(norms), np.percentile(norms, 95))
        with open(toeplitz_cache_path, "wb") as f:
            pkl.dump(toeplitz_quantiles, f)
    return toeplitz_quantiles[D]


def norm_bound_shots(x_norm_sq: float, h_sq: float, energy: float, g_norm: float, bias: float = 0.0) -> float:
    """Shots per estimate from |dE| <= ||x||^2 (sqrt<H^2> + |E|) ||G|| / sqrt(M) <= eps - |bias|."""
    if abs(bias) >= chemical_accuracy:
        return np.inf
    return (x_norm_sq * (np.sqrt(h_sq) + abs(energy)) * g_norm / (chemical_accuracy - abs(bias))) ** 2


def load_dataset_results(dataset: dict) -> dict:
    """All results_n*.pkl in a dataset directory, keyed by natoms."""
    out = {}
    for path in glob.glob(f"{dataset['dir']}/results_n*.pkl"):
        n = int(re.search(r"results_n(\d+)\.pkl", path).group(1))
        with open(path, "rb") as f:
            out[n] = pkl.load(f)
        assert out[n]["params"]["subspace_dimension"] == d_for(dataset, n), (path, out[n]["params"]["subspace_dimension"])
    return dict(sorted(out.items()))

In [42]:
# Per-dataset analysis at the simulated sizes (slowest step, ~1 min): Monte Carlo shot counts, first-order
# sigma_1^2, ||x||^2 and the norm bound. Saved to <dir>/bound_analysis.pkl.
z_samples_bound = np.random.default_rng(0).standard_normal(200_000)
dense_shots = np.logspace(nshots_min_exp, nshots_max_exp, 181)

for label, ds in bound_datasets.items():
    rows = []
    for n, res in load_dataset_results(ds).items():
        h, s, h_var, s_var, E0 = res["h"], res["s"], res["h_var"], res["s_var"], res["energy_exact"]
        d = res["params"]["subspace_dimension"]

        # Monte Carlo (the simulated shot counts).
        errs = np.abs(res["noisy_energies"] - E0)
        mc_median = shots_for_accuracy(res["nshots"], np.median(errs, axis=1), chemical_accuracy)
        mc_p95 = shots_for_accuracy(res["nshots"], np.percentile(errs, 95, axis=1), chemical_accuracy)

        # First order on a dense shot grid; evaluate everything at the threshold where the median reaches eps.
        med, p95 = [], []
        for M in dense_shots:
            E, s1sq, _ = first_order_sigma1_sq(h, s, h_var, s_var, noisy_threshold_for(res, M))
            e = np.abs((E - E0) + np.sqrt(s1sq / M) * z_samples_bound)
            med.append(np.median(e))
            p95.append(np.percentile(e, 95))
        fo_median = shots_for_accuracy(dense_shots, np.array(med), chemical_accuracy)
        thr = noisy_threshold_for(res, fo_median)
        E_star, s1sq_star, _ = first_order_sigma1_sq(h, s, h_var, s_var, thr)
        _, x = thresholded_ground(h, s, thr)
        x_norm_sq = np.vdot(x, x).real
        bias = E_star - E0
        h_sq = E0 ** 2 + h_var[0, 0].real  # <psi0|H^2|psi0> = E^2 + Var_psi0(H) (diagonal variance).
        g50, g95 = toeplitz_norm_quantiles(d + 1)

        rows.append({
            "natoms": n, "d": d, "E": E0, "krylov_error": res["subspace_error"],
            "mc_median": mc_median, "mc_p95": mc_p95,
            "fo_median": fo_median, "fo_p95": shots_for_accuracy(dense_shots, np.array(p95), chemical_accuracy),
            "threshold_star": thr, "sigma1sq_star": s1sq_star, "bias_star": bias,
            "x_norm_sq": x_norm_sq, "h_sq": h_sq, "var_psi0": h_var[0, 0].real,
            "bound_median": norm_bound_shots(x_norm_sq, h_sq, E0, g50, bias),
            "bound_p95": norm_bound_shots(x_norm_sq, h_sq, E0, g95, bias),
        })
    with open(f"{ds['dir']}/bound_analysis.pkl", "wb") as f:
        pkl.dump({"label": label, "dataset": ds, "chemical_accuracy": chemical_accuracy, "rows": rows}, f)

    print(f"== {label} ({ds['dir']}) ==")
    print(f"{'N':>2} {'d':>3} {'|x|^2':>6} {'1/(d+1)':>7} {'s1^2(d+1)/E^2':>13} {'bias mHa':>8} | "
          f"{'MC med':>8} {'1st med':>8} {'bnd med':>8} | {'MC 95':>8} {'1st 95':>8} {'bnd 95':>8}")
    for r in rows:
        print(f"{r['natoms']:>2} {r['d']:>3} {r['x_norm_sq']:6.3f} {1 / (r['d'] + 1):7.3f} "
              f"{r['sigma1sq_star'] * (r['d'] + 1) / r['E'] ** 2:13.3f} {r['bias_star'] * 1e3:8.2f} | "
              f"{r['mc_median']:8.1e} {r['fo_median']:8.1e} {r['bound_median']:8.1e} | "
              f"{r['mc_p95']:8.1e} {r['fo_p95']:8.1e} {r['bound_p95']:8.1e}")

== d = 20 (data) ==
 N   d  |x|^2 1/(d+1) s1^2(d+1)/E^2 bias mHa |   MC med  1st med  bnd med |    MC 95   1st 95   bnd 95
 3  20  0.048   0.048         0.152     1.32 |  3.0e+03  3.1e+03  1.6e+07 |  1.6e+06  2.7e+05  2.9e+07
 4  20  0.051   0.048         0.753     0.86 |  4.1e+04  3.5e+04  7.7e+06 |  5.1e+05  6.3e+05  1.4e+07
 5  20  0.051   0.048         0.733     0.69 |  4.7e+04  3.8e+04  6.1e+06 |  2.9e+05  6.2e+05  1.1e+07
 6  20  0.049   0.048         0.713     0.23 |  4.8e+04  6.2e+04  4.5e+06 |  5.9e+05  6.0e+05  8.2e+06
 7  20  0.050   0.048         0.743    -0.98 |  1.8e+05  9.0e+04  2.6e+07 |  5.1e+06  4.8e+06  4.7e+07
== d = 4N (data/d4N) ==
 N   d  |x|^2 1/(d+1) s1^2(d+1)/E^2 bias mHa |   MC med  1st med  bnd med |    MC 95   1st 95   bnd 95
 3  12  0.077   0.077         0.073     1.63 |  2.1e+04  7.2e+04      inf |  3.8e+05  5.3e+05      inf
 4  16  0.061   0.059         0.762     1.08 |  5.0e+04  4.9e+04  1.8e+07 |  8.2e+05  1.3e+06  3.3e+07
 5  20  0.051   0.048        

In [43]:
# Fits and extrapolated curves for each dataset. Saved to <dir>/bound_extrapolation.pkl.
with open("data/ccsdt_energies.pkl", "rb") as f:
    ccsdt_energies_bound = pkl.load(f)
N_curve = np.arange(3, bound_extrapolate_natoms + 1)

for label, ds in bound_datasets.items():
    with open(f"{ds['dir']}/bound_analysis.pkl", "rb") as f:
        rows = pkl.load(f)["rows"]
    fit = [r for r in rows if r["natoms"] >= bound_fit_natoms_min]
    N_fit = np.array([r["natoms"] for r in fit])
    s1_fit = np.array([r["sigma1sq_star"] for r in fit])
    kappa_d = np.array([r["sigma1sq_star"] * (r["d"] + 1) / r["E"] ** 2 for r in fit])
    kappa = kappa_d.mean()
    powerlaw = np.polyfit(np.log(N_fit), np.log(s1_fit), 1)
    exponential = np.polyfit(N_fit, np.log(s1_fit), 1)

    # E(N): simulated (exact, truncated H) where available, CCSD(T) (full H) beyond.
    E_of_N = dict(ccsdt_energies_bound)
    E_of_N.update({r["natoms"]: r["E"] for r in rows})
    var_of_N = {r["natoms"]: r["var_psi0"] for r in rows}
    E_curve = np.array([E_of_N[n] for n in N_curve])
    d_curve = np.array([d_for(ds, n) for n in N_curve])

    sigma1sq_models = {
        "kappa": kappa * E_curve ** 2 / (d_curve + 1),
        "powerlaw": np.exp(np.polyval(powerlaw, np.log(N_curve))),
        "exponential": np.exp(np.polyval(exponential, N_curve)),
    }
    shots = {f"{m}_{stat}": q ** 2 * v / chemical_accuracy ** 2
             for m, v in sigma1sq_models.items() for stat, q in [("median", Q50), ("p95", Q95)]}
    for stat, k in [("median", 0), ("p95", 1)]:
        shots[f"bound_{stat}"] = np.array([
            norm_bound_shots(1 / (d + 1), E ** 2 + var_of_N.get(n, var_psi0_extrap), E, toeplitz_norm_quantiles(d + 1)[k])
            for n, d, E in zip(N_curve, d_curve, E_curve)])

    with open(f"{ds['dir']}/bound_extrapolation.pkl", "wb") as f:
        pkl.dump({"label": label, "dataset": ds, "N": N_curve, "d": d_curve, "E": E_curve,
                  "estimates": 1 + 4 * d_curve, "kappa": kappa, "kappa_spread": kappa_d.std(),
                  "powerlaw": powerlaw, "exponential": exponential, "fit_natoms": N_fit,
                  "sigma1sq": sigma1sq_models, "shots_per_estimate": shots}, f)

    i = -1
    print(f"== {label}: kappa = sigma1^2 (d+1)/E^2 = {kappa:.3f} +/- {kappa_d.std():.3f}, "
          f"power law N^{powerlaw[0]:.2f}, exp({exponential[0]:.3f} N) ==")
    print(f"   N = {N_curve[i]}, d = {d_curve[i]}, {1 + 4 * d_curve[i]} estimates (zero bias)")
    for m in ["kappa", "powerlaw", "exponential", "bound"]:
        a, b = shots[f"{m}_median"][i], shots[f"{m}_p95"][i]
        print(f"   {m:12s} per estimate {a:9.2e} / {b:9.2e}   total {a * (1 + 4 * d_curve[i]):9.2e} / {b * (1 + 4 * d_curve[i]):9.2e}  (median / 95%)")

== d = 20: kappa = sigma1^2 (d+1)/E^2 = 0.735 +/- 0.015, power law N^1.74, exp(0.327 N) ==
   N = 22, d = 20, 81 estimates (zero bias)
   kappa        per estimate  8.69e+05 /  7.34e+06   total  7.04e+07 /  5.94e+08  (median / 95%)
   powerlaw     per estimate  5.38e+05 /  4.54e+06   total  4.36e+07 /  3.68e+08  (median / 95%)
   exponential  per estimate  1.03e+07 /  8.67e+07   total  8.32e+08 /  7.02e+09  (median / 95%)
   bound        per estimate  4.32e+07 /  7.76e+07   total  3.50e+09 /  6.29e+09  (median / 95%)
== d = 4N: kappa = sigma1^2 (d+1)/E^2 = 0.734 +/- 0.023, power law N^0.73, exp(0.156 N) ==
   N = 22, d = 88, 353 estimates (zero bias)
   kappa        per estimate  2.05e+05 /  1.73e+06   total  7.22e+07 /  6.10e+08  (median / 95%)
   powerlaw     per estimate  1.20e+05 /  1.01e+06   total  4.24e+07 /  3.58e+08  (median / 95%)
   exponential  per estimate  5.74e+05 /  4.84e+06   total  2.03e+08 /  1.71e+09  (median / 95%)
   bound        per estimate  1.52e+07 /  2.36e+07

In [44]:
# sigma_1^2 (d+1) / E^2 for both datasets (the generalized kappa). Reads only the saved analyses.
with plt.rc_context(bound_rc):
    fig, ax = plt.subplots(figsize=(6.0, 3.9))
    offsets = np.linspace(-0.08, 0.08, len(bound_datasets))  # Datasets can coincide (same d at some N).
    for (label, ds), color, marker, dx in zip(bound_datasets.items(), [INK, VIOLET], ["o", "s"], offsets):
        with open(f"{ds['dir']}/bound_analysis.pkl", "rb") as f:
            rows = pkl.load(f)["rows"]
        with open(f"{ds['dir']}/bound_extrapolation.pkl", "rb") as f:
            kappa = pkl.load(f)["kappa"]
        N_all = np.array([r["natoms"] for r in rows])
        k_all = np.array([r["sigma1sq_star"] * (r["d"] + 1) / r["E"] ** 2 for r in rows])
        fitted = N_all >= bound_fit_natoms_min
        ax.plot(N_all[fitted] + dx, k_all[fitted], marker, color=color, ms=8, mec="white", mew=1.5, zorder=5,
                label=f"{label}: mean (N $\\geq$ {bound_fit_natoms_min}) = {kappa:.3f}")
        ax.plot(N_all[~fitted] + dx, k_all[~fitted], marker, mfc="white", mec=color, ms=8, zorder=5)
        ax.axhline(kappa, color=color, lw=1, ls=":")
    ax.set_ylim(0, None)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("Number of H atoms, N")
    ax.set_ylabel("$\\sigma_1^2\\,(d+1) / E^2$")
    ax.set_title("Generalized $\\kappa$ (hollow: excluded from fits)", loc="left", fontsize=12)
    ax.legend(fontsize=8, frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig("data/kappa_both_datasets.pdf")

In [45]:
# Simulated shot counts and all extrapolations, for both datasets. Reads only the saved analyses (fast).
from matplotlib.lines import Line2D

models = [("kappa", "$\\kappa E^2/(d+1)$ (first order)", BLUE),
          ("powerlaw", "Power law (first order)", ORANGE),
          ("exponential", "Exponential (first order)", AQUA),
          ("bound", "Norm bound, $\\|x\\|^2 = 1/(d+1)$", VIOLET)]

with plt.rc_context(bound_rc):
    fig, axes = plt.subplots(len(bound_datasets), 2, figsize=(11.5, 4.1 * len(bound_datasets)), sharex=True, squeeze=False)
    for row_axes, (label, ds) in zip(axes, bound_datasets.items()):
        with open(f"{ds['dir']}/bound_analysis.pkl", "rb") as f:
            rows = pkl.load(f)["rows"]
        with open(f"{ds['dir']}/bound_extrapolation.pkl", "rb") as f:
            ex = pkl.load(f)
        N_sim = [r["natoms"] for r in rows]
        for ax, total in zip(row_axes, [False, True]):
            scale_curve = ex["estimates"] if total else np.ones_like(ex["N"])
            scale_sim = np.array([1 + 4 * r["d"] for r in rows]) if total else np.ones(len(rows))
            for key, _, color in models:
                ax.plot(ex["N"], ex["shots_per_estimate"][f"{key}_median"] * scale_curve, color=color, lw=2)
                ax.plot(ex["N"], ex["shots_per_estimate"][f"{key}_p95"] * scale_curve, color=color, lw=2, ls="--")
            ax.plot(N_sim, [r["bound_median"] for r in rows] * scale_sim, "o", mfc="white", mec=VIOLET, mew=1.8, ms=7, zorder=4)
            ax.plot(N_sim, [r["bound_p95"] for r in rows] * scale_sim, "^", mfc="white", mec=VIOLET, mew=1.8, ms=7, zorder=4)
            ax.plot(N_sim, [r["mc_median"] for r in rows] * scale_sim, "o", color=INK, ms=8, mec="white", mew=1.5, zorder=5)
            ax.plot(N_sim, [r["mc_p95"] for r in rows] * scale_sim, "^", color=INK, ms=8, mec="white", mew=1.5, zorder=5)
            ax.axvline(bound_extrapolate_natoms, color=MUTED, ls=":", lw=1)
            ax.set_yscale("log")
            ax.set_xticks(sorted(set(N_sim) | set(range(10, bound_extrapolate_natoms + 1, 4)) | {bound_extrapolate_natoms}))
            what = "Total shots, (1 + 4d) estimates" if total else "Shots per matrix-element estimate"
            ax.set_title(f"{label}: {what.lower()}", loc="left", fontsize=11)
            ax.set_ylabel("Total shots" if total else "Shots per estimate")
        for ax in row_axes:
            ax.set_ylim(1e3 if ax is row_axes[0] else 1e4, None)
    for ax in axes[-1]:
        ax.set_xlabel("Number of H atoms, N")

    handles = [Line2D([], [], color=c, lw=2, label=name) for _, name, c in models]
    handles += [Line2D([], [], color=MUTED, lw=2, label="Extrapolation, median"),
                Line2D([], [], color=MUTED, lw=2, ls="--", label="Extrapolation, 95% of runs"),
                Line2D([], [], color=INK, marker="o", ls="", ms=7, label="Simulated (Monte Carlo), median"),
                Line2D([], [], color=INK, marker="^", ls="", ms=7, label="Simulated (Monte Carlo), 95% of runs"),
                Line2D([], [], color=VIOLET, marker="o", mfc="white", ls="", ms=7, label="Norm bound, actual $\\|x\\|^2$ and bias, median"),
                Line2D([], [], color=VIOLET, marker="^", mfc="white", ls="", ms=7, label="Norm bound, actual $\\|x\\|^2$ and bias, 95%")]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=7.5, frameon=False)
    fig.suptitle(f"Shots for chemical accuracy ({chemical_accuracy * 1e3:.1f} mHa): simulations and extrapolations "
                 f"(fits use N $\\geq$ {bound_fit_natoms_min}; extrapolations assume zero bias)", x=0.01, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0.09, 1, 0.97))
    fig.savefig("data/shots_extrapolation_comparison.pdf")